In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Creating Agents with the OpenAI Agents SDK
---
# Code Interpreter Tool

* `CodeInterpreterTool` allows agents to write and execute Python 
    * OpenAI-hosted sandbox created on the fly or in advance
* Model decides code to write, executes it, and can auto-revise on errors
* Any generated files returned as container file citations in the result
---

## Imports and Constants

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display
from openai import OpenAI
from agents import Agent, CodeInterpreterTool, ModelSettings, Runner, trace

OUTPUT_DIR = Path('resources') / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI()

---

## Helper Function to Save Files Created in the Hosted Container
* Generated files come back as `container_file_citation` annotations in `result.new_items`
* Download artifacts promptly — hosted code-interpreter containers are ephemeral
    * Removed after 20 minutes of inactivity

In [ ]:
def save_container_artifacts(result, prefix):
    """Download Code Interpreter output files and display any images."""
    paths = []
    seen_files = set()
    number = 1

    # CodeInterpreterTool creates files in an OpenAI-hosted container. 
    # When final answer cites a generated file, SDK exposes it as a
    # container_file_citation annotation in result.new_items.
    for item in result.new_items:
        raw_item = getattr(item, 'raw_item', None)

        # Different run items have different shapes. getattr keeps this loop
        # safe across messages, tool calls and other item types.
        for content_part in getattr(raw_item, 'content', []) or []:
            for annotation in getattr(content_part, 'annotations', []) or []:
                # skip annotations that are not generated container files
                if getattr(annotation, 'type', None) != 'container_file_citation':
                    continue

                # same file can be cited more than once in final output
                file_key = (annotation.container_id, annotation.file_id)
                
                if file_key not in seen_files:
                    seen_files.add(file_key) 

                # preserve generated file extension when possible
                suffix = Path(annotation.filename).suffix or '.bin'
                path = OUTPUT_DIR / f'{prefix}_{number}{suffix}'

                # download file bytes from hosted container
                data = client.containers.files.content.retrieve(
                    file_id=annotation.file_id, 
                    container_id=annotation.container_id
                )

                # save the file
                path.write_bytes(data.read())
                paths.append(path)
                number += 1

                # display image artifacts inline for immediate feedback
                if path.suffix.lower() in ('.png', '.jpg', '.jpeg'):
                    display(Image(filename=str(path), width=400))

    return paths


def display_code_interpreter_code(result):
    """Display Python code written by Code Interpreter tool calls."""
    code_blocks = []

    for item in result.new_items:
        raw_item = getattr(item, 'raw_item', None)

        if getattr(raw_item, 'type', None) == 'code_interpreter_call':
            code = getattr(raw_item, 'code', None)

            if code:
                code_blocks.append(f'```python\n{code}\n```')

    if code_blocks:
        display(Markdown('## Code Interpreter Generated Code\n\n' + '\n\n'.join(code_blocks)))
    else:
        display(Markdown('_No Code Interpreter code found in this result._'))

---
## Demo: Benchmark Python Implementations with Code Interpreter

* Compare a `for` loop, a list comprehension, and NumPy for the same computation
* Code Interpreter runs repeatable `timeit` measurements in the hosted sandbox
* The agent generates a bar chart and returns a Markdown table of results
* `ModelSettings(tool_choice='required')` forces the agent to use the tool rather than answering from training data

In [ ]:
benchmark_agent = Agent(
    name='Python Benchmark Agent',
    model='gpt-5.6-terra',
    instructions="""You benchmark Python implementations.
        Use Code Interpreter for all timing measurements.
        Return concise results.""",
    model_settings=ModelSettings(tool_choice='required'), # force tool use
    tools=[
        CodeInterpreterTool(
            tool_config={
                'type': 'code_interpreter',
                'container': {'type': 'auto'} # OpenAI creates hosted container automatically
            }
        )
    ]
)

prompt = """Use Code Interpreter to benchmark three ways to compute
the sum of squares from 0 to 999,999: a for loop, a list comprehension,
and NumPy. Use timeit for stable timing. Analyze the results, including
a Markdown table summary, and save a bar chart as benchmark.png."""

with trace('02-05-04-code-interpreter-benchmark'):
    result = await Runner.run(benchmark_agent, prompt)

display(Markdown(result.final_output))
save_container_artifacts(result, 'benchmark')
display_code_interpreter_code(result)

---

## Visualizing the Agent Workflow

In [ ]:
from agents.extensions.visualization import draw_graph
draw_graph(benchmark_agent, filename='agent_workflow.png')

---

## Documentation References

* [OpenAI Code Interpreter Guide](https://developers.openai.com/api/docs/guides/tools-code-interpreter)
* [OpenAI Agents SDK: Tools](https://openai.github.io/openai-agents-python/tools/)

---
&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved.